In [1]:
import pandas as pd
import numpy as np

In [2]:
# Baixando a base de alunos

df = pd.read_parquet('../data/alunos.parquet')
df.head()

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,chave_rede,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
0,2023,1302603,Manaus,60000666,13045431,1,2° ano do Ensino Fundamental,3,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
1,2023,1302306,Jutaí,60001237,13014346,1,2° ano do Ensino Fundamental,3,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
2,2023,1504703,Moju,60002245,15060424,1,2° ano do Ensino Fundamental,3,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
3,2023,1501402,Belém,60003343,15016961,1,2° ano do Ensino Fundamental,3,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN
4,2023,2102903,Carutapera,60004621,21023834,1,2° ano do Ensino Fundamental,3,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN


In [3]:
# Tamanho da base
df.shape

(3867999, 14)

In [4]:
# Checando os dados em branco
df.isna().sum()

ano                           0
id_municipio                  0
id_municipio_nome             0
id_escola                     0
id_aluno                      0
caderno                       0
serie                         0
chave_rede                    0
rede                          0
presenca                      0
preenchimento_caderno         0
alfabetizado                  0
proficiencia             513338
peso_aluno               513338
dtype: int64

In [5]:
# Conferir se tem notas de outras séries
df['serie'].value_counts()


serie
2° ano do Ensino Fundamental    3867999
Name: count, dtype: int64

In [6]:
df['ano'].value_counts()


ano
2024    2120560
2023    1747439
Name: count, dtype: Int64

In [7]:
# Conferir valores na coluna presenca
df['presenca'].value_counts()

presenca
Presente    3355846
Ausente      512153
Name: count, dtype: int64

In [8]:
alunos_presentes = df[df['presenca']=="Presente"]
qtd_alunos_presentes = len(alunos_presentes)
qtd_alunos_presentes

3355846

In [9]:
# Conferir coluna alfabetizado
df['alfabetizado'].isna().sum()

np.int64(0)

In [10]:
df['alfabetizado'].unique()

array(['Não', 'Sim'], dtype=object)

In [11]:
df['alfabetizado'].value_counts()

alfabetizado
Sim    1984546
Não    1883453
Name: count, dtype: int64

Nossa variável alvo não tem dados em branco e possui um bom equilíbrio entre alfabetizados e não alfabetizados, o que será importante para o modelo.

In [12]:
# Conferindo se tem algum aluno ausente alfabetizado
len(df.query("presenca == 'Ausente' and alfabetizado == 'Sim'"))

0

In [13]:
df[df['presenca'] == 'Ausente']['alfabetizado'].value_counts()

alfabetizado
Não    512153
Name: count, dtype: int64

In [14]:
# Conferir a coluna preenchimento_caderno
df['preenchimento_caderno'].isna().sum()

np.int64(0)

In [15]:
df['preenchimento_caderno'].value_counts()

preenchimento_caderno
Prova preenchida        3354661
Prova não preenchida     513338
Name: count, dtype: int64

In [16]:
df[df['preenchimento_caderno'] == 'Prova não preenchida']['alfabetizado'].value_counts()

alfabetizado
Não    513338
Name: count, dtype: int64

In [17]:
# Quantidade de alunos que compareceram mas não preencheram o caderno de prova
qtd_alunos_presentes_nao_preenchida = len(df.query("presenca == 'Presente' and preenchimento_caderno == 'Prova não preenchida'"))
percentual_presentes_nao_preenchida = qtd_alunos_presentes_nao_preenchida/qtd_alunos_presentes *100
print(f'A quantidade de alunos que compareceu à prova e não preencheu o caderno corresponde a {round(percentual_presentes_nao_preenchida, 2)}%')



A quantidade de alunos que compareceu à prova e não preencheu o caderno corresponde a 0.04%


Verificamos que todos os alunos ausentes são considerados analfabetos, assim como os alunos que não preencheram o caderno de prova. 

Existe, portanto, uma dependência determinística nos dados, ou seja, a alfabetização desse grupo de alunos é determinada por uma regra lógica pré-existente, e não por um padrão latente nas características socioeconômicas ou territoriais do aluno.

Por esta razão, iremos trabalhar apenas com os alunos presentes e que preencheram a prova, para evitar vazamento de dados, uma vez que o modelo passará a tentar prever a presença na prova e não a competência de alfabetização.

In [18]:
# Criando o dataframe apenas com os alunos que compareceram e preencheram o caderno de prova.

alunos_presentes_prova_preenchida = alunos_presentes.query('preenchimento_caderno == "Prova preenchida"')
alunos_presentes_prova_preenchida


,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,chave_rede,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
3679,2023,3302403,Macaé,60023754,33043575,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,769.246600,0.228073
3680,2023,3302403,Macaé,60023754,33043562,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,768.038400,0.228073
3681,2023,3302403,Macaé,60023754,33043577,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Não,716.986000,0.228073
3682,2023,3302403,Macaé,60023754,33043574,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Não,659.301816,0.228073
3683,2023,3303500,Nova Iguaçu,60024742,33059755,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Não,679.834400,0.306015
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3867994,2023,5108105,Tesouro,60041115,51033883,9,2° ano do Ensino Fundamental,2,Estadual,Presente,Prova preenchida,Não,711.650000,6.520000
3867995,2023,5108105,Tesouro,60041115,51033885,9,2° ano do Ensino Fundamental,2,Estadual,Presente,Prova preenchida,Não,657.990000,6.520000
3867996,2023,3122306,Divinópolis,60018848,31000615,9,2° ano do Ensino Fundamental,2,Estadual,Presente,Prova preenchida,Sim,778.355700,6.839797
3867997,2023,3131307,Ipatinga,60020707,31000571,9,2° ano do Ensino Fundamental,2,Estadual,Presente,Prova preenchida,Sim,825.526567,7.027251


In [19]:
alunos_presentes_prova_preenchida['proficiencia'].isna().sum()

np.int64(0)

In [20]:
# Verificar a nota mínima para consideradar o aluno alfabetizado
alunos_presentes_prova_preenchida[alunos_presentes_prova_preenchida['alfabetizado'] == 'Sim']['proficiencia'].min()

np.float64(743.0)

Verificamos que os alunos alfabetizados possuem proficiência maior do que 743, conforme a regra definida.

In [21]:
# Precisamos conferir se o id_escola se mantem de um ano para outro.

alunos_presentes_prova_preenchida['id_escola'].nunique()

42802

In [22]:
print(alunos_presentes_prova_preenchida[alunos_presentes_prova_preenchida['ano']==2023]['id_escola'].nunique())
print(alunos_presentes_prova_preenchida[alunos_presentes_prova_preenchida['ano']==2024]['id_escola'].nunique())

36525
42328


In [23]:
municipios_por_escola = alunos_presentes_prova_preenchida.groupby('id_escola')['id_municipio'].nunique()
escolas_com_divergencia = municipios_por_escola[municipios_por_escola > 1].index

print(f"Total de escolas com divergência: {len(escolas_com_divergencia)}")

Total de escolas com divergência: 35187


Ficou claro que o id_escola é um número atribuído aleatoriamente a cada ano. Ou seja, não tem como fazer métricas por id_escola para mais de um ano. Então, a média de alfabetizados de uma escola precisa ser feita a cada ano. 
Porém, não devemos usar esse dado de percentual de alfabetizado por escola, porque não podemos usar o dado do ano para treinar o modelo do mesmo ano.

Decidimos também usar apenas os dados de 2024, pois, usar métricas de 2023 (taxa_alfabetização_municipio_2023, inse_2023, etc.) para prever o desempenho do aluno em 2024 garante que a informação municipal/escolar é verdadeiramente um antecedente histórico, e não um cálculo feito com os mesmos alunos que estão sendo testados. 

Além disso, Simula o cenário real de aplicação (Deploy): No início do ano letivo de 2024, os gestores públicos só possuem os dados históricos consolidados até 2023 para tentar identificar com antecedência quais alunos correm risco de não serem alfabetizados.

In [24]:
# Antes de fazer a junção, vamos filtrar apenas os alunos de 2024.
alunos_presentes_prova_preenchida_2024 = alunos_presentes_prova_preenchida[alunos_presentes_prova_preenchida['ano']==2024]
alunos_presentes_prova_preenchida_2024.drop(columns='ano', inplace=True)
len(alunos_presentes_prova_preenchida_2024)

C:\Users\thiag\AppData\Local\Temp\ipykernel_10512\279382831.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  alunos_presentes_prova_preenchida_2024.drop(columns='ano', inplace=True)


1851852

In [25]:
alunos_presentes_prova_preenchida_2024

,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,chave_rede,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno
5766,1100452,Buritis,60000332,11020232,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,764.590000,1.000000
5767,2207702,Parnaíba,60006401,22001819,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Não,729.290000,1.000000
5768,2304400,Fortaleza,60008121,23030799,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,824.310000,1.000000
5769,2304400,Fortaleza,60008100,23089103,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,779.250000,1.000000
5770,2311306,Quixadá,60008319,23015579,1,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,809.620000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3867972,4208450,Itapoá,60034848,42041289,9,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Não,687.130000,3.090000
3867973,1505064,Novo Repartimento,60002621,15051448,9,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Não,689.450000,3.100000
3867975,4208450,Itapoá,60034849,42041358,9,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,781.770000,3.140000
3867979,4209102,Joinville,60034798,42096288,9,2° ano do Ensino Fundamental,3,Municipal,Presente,Prova preenchida,Sim,757.760000,3.210000


In [26]:
# Partindo para a junção das tabelas de alunos e municípios
df_municipio = pd.read_parquet('../data/dados_modelo/municipio_eng_feat.parquet')
df_municipio.head()


,ano,id_municipio,id_municipio_nome,serie,chave_rede,rede,taxa_alfabetizacao_mun,media_portugues,sigla_uf,taxa_alfabetizacao_uf,indice_analf,MEDIA_INSE,razao_inse_analf,raz_alfab_mun_uf,razao_taxa_uf_inse
0,<NA>,5101837,None,None,None,None,NaN,NaN,MT,NaN,8.46,NaN,NaN,NaN,NaN
1,2023,1100205,Porto Velho,2° ano do Ensino Fundamental,2,Estadual,57.45,747.2139,RO,58.65,4.36,4.959,1.137383,0.979540,11.824597
2,2023,1100205,Porto Velho,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),64.47,760.4047,RO,64.60,4.36,4.892,1.122016,0.997988,13.202534
3,2023,1100205,Porto Velho,2° ano do Ensino Fundamental,3,Municipal,65.06,761.5118,RO,65.17,4.36,4.789,1.098392,0.998312,13.605428
7,2023,1100338,Nova Mamoré,2° ano do Ensino Fundamental,3,Municipal,63.02,754.3625,RO,65.17,9.31,4.733,0.508378,0.967009,13.766371


In [29]:
# Vamos pegar apenas as colunas que serão mantidas no dataframe resultante da agregação.

colunas_mantidas = ['ano','id_municipio','taxa_alfabetizacao_mun','media_portugues', 'chave_rede',
                    'sigla_uf','taxa_alfabetizacao_uf','indice_analf','MEDIA_INSE', 
                    'razao_inse_analf', 'raz_alfab_mun_uf','razao_taxa_uf_inse']

df_com_dados_agregados = pd.merge(
    alunos_presentes_prova_preenchida_2024, df_municipio[colunas_mantidas],
    on=['id_municipio', 'chave_rede'],
    how='right'
)

print('Tamanho dataset antes do merge: ', len(alunos_presentes_prova_preenchida_2024))
print('Tamanho dataset depois do merge: ', len(df_com_dados_agregados))

Tamanho dataset antes do merge:  1851852
Tamanho dataset depois do merge:  1821410


In [30]:
# Vamos analisar a quantidade de dados nulos na base que será enviada para o modelo

nulos_qtd = df_com_dados_agregados.isna().sum()
nulos_pct = (nulos_qtd / len(df_com_dados_agregados)) * 100
df_relatorio_nulos = pd.DataFrame({
    'coluna': nulos_qtd.index,
    'qtd_nulos': nulos_qtd.values,
    'pct_nulos': nulos_pct.values
})

df_relatorio_nulos = (
    df_relatorio_nulos
    .sort_values(by='qtd_nulos', ascending=False)
    .reset_index(drop=True)
)

df_relatorio_nulos['pct_nulos'] = df_relatorio_nulos['pct_nulos'].map('{:.2f}%'.format)

print(df_relatorio_nulos)


                    coluna  qtd_nulos pct_nulos
0        id_municipio_nome       5140     0.28%
1                id_escola       5140     0.28%
2                 id_aluno       5140     0.28%
3                    serie       5140     0.28%
4                  caderno       5140     0.28%
5                 presenca       5140     0.28%
6                     rede       5140     0.28%
7             proficiencia       5140     0.28%
8               peso_aluno       5140     0.28%
9    preenchimento_caderno       5140     0.28%
10            alfabetizado       5140     0.28%
11              MEDIA_INSE       1467     0.08%
12        razao_inse_analf       1467     0.08%
13      razao_taxa_uf_inse       1467     0.08%
14              chave_rede         21     0.00%
15   taxa_alfabetizacao_uf         21     0.00%
16                     ano         21     0.00%
17  taxa_alfabetizacao_mun         21     0.00%
18         media_portugues         21     0.00%
19        raz_alfab_mun_uf         21   

O restante dos dados nulos serão tratados em momento posterior.

In [32]:
# Exclusão de Colunas Irrelevantes
cols_inuteis = ['id_aluno', 'caderno', 'preenchimento_caderno', 'presenca', 'serie', 'rede', 'id_municipio_nome']
base_modelo = df_com_dados_agregados.drop(columns=cols_inuteis)

In [33]:
# Salvar base 
base_modelo.to_parquet('../data/dados_modelo/base_modelo.parquet')